# Tutorial: Milestone 4 Export/Import + Cache (Rust Backend)

Audience:
- Engineers validating Milestone 4 parity behavior against the real Rust daemon.

Prerequisites:
- Run `uv sync` in this repository.
- Rust toolchain and `cargo` are available.

Learning goals:
- Validate `POST /export/viewstate` and `POST /import/viewstate` behavior through the Python client against the Rust backend.
- Confirm import rebasing (`view_id`, `session_id`, `state_version`, `state_hash`) and compat-session behavior.
- Exercise low/high render cache budget hints in real usage and verify render stability.


## Step 1 - Imports and helpers


In [1]:
from __future__ import annotations

import base64
import copy
import os
import shutil
import sys
import tempfile
from io import BytesIO
from pathlib import Path

from PIL import Image

from lucida.client import LucidaClient, LucidaClientError

cwd = Path.cwd().resolve()
if (cwd / 'MIGRATION.md').exists():
    REPO_ROOT = cwd
elif (cwd.parent.parent / 'MIGRATION.md').exists():
    REPO_ROOT = cwd.parent.parent
else:
    raise AssertionError('Could not locate repository root containing MIGRATION.md')

sys.path.insert(0, str(REPO_ROOT / 'tests'))
from rust_daemon import start_rust_daemon  # noqa: E402
from parity.data_setup import create_render_omezarr  # noqa: E402


def decode_size(payload_b64: str) -> tuple[int, int]:
    image = Image.open(BytesIO(base64.b64decode(payload_b64))).convert('RGBA')
    return image.size


def expect_client_error(fn, expected_fragment: str) -> None:
    try:
        fn()
    except LucidaClientError as exc:
        message = str(exc)
        assert expected_fragment in message, message
        print('Observed expected error:', expected_fragment)
        return
    raise AssertionError(f'Expected LucidaClientError containing {expected_fragment!r}.')


## Step 2 - Build fixture dataset and start a real Rust daemon


In [2]:
tmp_dir = Path(tempfile.mkdtemp(prefix='lucida-m4-rust-'))
dataset_uri = create_render_omezarr(str(tmp_dir / 'render-demo.zarr'))

rust_daemon = start_rust_daemon(repo_root=REPO_ROOT, env=dict(os.environ))
client = LucidaClient(base_url=rust_daemon.base_url, backend='rust')

print('dataset_uri:', dataset_uri)
print('rust_base_url:', rust_daemon.base_url)


dataset_uri: /var/folders/hs/qw7ws1q52153c4c639t_p3600000gn/T/lucida-m4-rust-u8fznu38/render-demo.zarr
rust_base_url: http://127.0.0.1:65444


## Step 3 - Create a source view and export its persisted state


In [3]:
source_session = client.create_session()
opened = client.open_dataset(uri=dataset_uri, session_id=source_session.session_id)
created = client.create_view(
    dataset_id=opened.dataset_summary.dataset_id,
    session_id=source_session.session_id,
    mode='2d',
)

exported = client.export_viewstate(
    view_id=created.view_state.view_id,
    session_id=source_session.session_id,
)

assert exported.export_id.startswith('exp_')
assert exported.source_view_id == created.view_state.view_id
assert exported.view_state.view_id == created.view_state.view_id
assert exported.view_state.state_hash

print('source_session:', source_session.session_id)
print('source_view:', created.view_state.view_id)
print('export_id:', exported.export_id)


source_session: session_9913af34916b4d1f
source_view: view_bb2db3d925844fbe
export_id: exp_75ba752d00b54bb4


## Step 4 - Import into same session, new session, and compat session


In [4]:
imported_same = client.import_viewstate(
    view_state=exported.view_state,
    session_id=source_session.session_id,
)
assert imported_same.import_id.startswith('imp_')
assert imported_same.imported_from_view_id == created.view_state.view_id
assert imported_same.view_state.view_id != created.view_state.view_id
assert imported_same.view_state.session_id == source_session.session_id
assert imported_same.view_state.state_version == 0
assert imported_same.view_state.state_hash
assert imported_same.selectors_applied

new_session = client.create_session()
imported_new = client.import_viewstate(
    view_state=exported.view_state,
    session_id=new_session.session_id,
)
assert imported_new.view_state.session_id == new_session.session_id
assert imported_new.view_state.view_id not in {
    created.view_state.view_id,
    imported_same.view_state.view_id,
}

imported_compat = client.import_viewstate(view_state=exported.view_state)
assert imported_compat.view_state.session_id.startswith('compat_')

print('same_session_import:', imported_same.view_state.view_id)
print('new_session_import:', imported_new.view_state.view_id)
print('compat_session_import:', imported_compat.view_state.session_id)


same_session_import: view_040a73eb912d4a80
new_session_import: view_daff1770e28241c6
compat_session_import: compat_a107aab420984682


## Step 5 - Validate import error contracts


In [5]:
source_state = exported.view_state.model_dump(mode='json')

missing_dataset_state = copy.deepcopy(source_state)
missing_dataset_state['datasets'][0]['dataset_id'] = 'ds_missing'
for layer in missing_dataset_state['layers']:
    if layer.get('dataset_id') is not None:
        layer['dataset_id'] = 'ds_missing'
expect_client_error(
    lambda: client.import_viewstate(
        view_state=missing_dataset_state,
        session_id=source_session.session_id,
    ),
    'dataset_not_found',
)

unsupported_mode_state = copy.deepcopy(source_state)
unsupported_mode_state['mode'] = '3d'
unsupported_mode_state['view_3d'] = {}
expect_client_error(
    lambda: client.import_viewstate(
        view_state=unsupported_mode_state,
        session_id=source_session.session_id,
    ),
    'unsupported_mode',
)

layer_mismatch_state = copy.deepcopy(source_state)
layer_mismatch_state['layers'][0]['dataset_id'] = 'ds_other'
expect_client_error(
    lambda: client.import_viewstate(
        view_state=layer_mismatch_state,
        session_id=source_session.session_id,
    ),
    'invalid_viewstate_import',
)


Observed expected error: dataset_not_found
Observed expected error: unsupported_mode
Observed expected error: invalid_viewstate_import


## Step 6 - Exercise low/high cache budget hints with repeated renders


In [6]:
source_before = client.get_view(view_id=created.view_state.view_id).view_state

client.update_view(
    view_id=created.view_state.view_id,
    patch=[
        {
            'op': 'replace',
            'path': '/performance',
            'value': {
                'max_cpu_cache_bytes': 4096,
                'max_gpu_cache_bytes': 16384,
            },
        }
    ],
)
state_after_low_config = client.get_view(view_id=created.view_state.view_id).view_state

render_low_1 = client.render_image(
    view_id=created.view_state.view_id,
    width_px=96,
    height_px=64,
)
render_low_2 = client.render_image(
    view_id=created.view_state.view_id,
    width_px=96,
    height_px=64,
)

assert render_low_1.status == 'ok'
assert render_low_2.status == 'ok'
assert render_low_1.view_id == created.view_state.view_id
assert render_low_2.view_id == created.view_state.view_id
assert decode_size(render_low_1.images[0].bytes_base64 or '') == (96, 64)
assert decode_size(render_low_2.images[0].bytes_base64 or '') == (96, 64)

state_after_low_renders = client.get_view(view_id=created.view_state.view_id).view_state
assert state_after_low_renders.state_version == state_after_low_config.state_version
assert state_after_low_renders.state_hash == state_after_low_config.state_hash

client.update_view(
    view_id=created.view_state.view_id,
    patch=[
        {
            'op': 'replace',
            'path': '/performance',
            'value': {
                'max_cpu_cache_bytes': 1_048_576,
                'max_gpu_cache_bytes': 2_097_152,
            },
        }
    ],
)
state_after_high_config = client.get_view(view_id=created.view_state.view_id).view_state

render_high = client.render_image(
    view_id=created.view_state.view_id,
    width_px=96,
    height_px=64,
)
assert render_high.status == 'ok'
assert decode_size(render_high.images[0].bytes_base64 or '') == (96, 64)

state_after_high_render = client.get_view(view_id=created.view_state.view_id).view_state
assert state_after_high_render.state_version == state_after_high_config.state_version
assert state_after_high_render.state_hash == state_after_high_config.state_hash
assert state_after_high_render.state_version >= source_before.state_version

print('low_budget_render_ids:', render_low_1.render_id, render_low_2.render_id)
print('high_budget_render_id:', render_high.render_id)


low_budget_render_ids: ren_e31937034b6c421f ren_2dfc81e998114e4d
high_budget_render_id: ren_fc7a571898024355


## Step 7 - Cleanup


In [7]:
rust_daemon.stop()
shutil.rmtree(tmp_dir, ignore_errors=True)
print('Cleaned temporary dataset and stopped Rust daemon.')


Cleaned temporary dataset and stopped Rust daemon.
